# Nexus — train EMERALD on Crafter (A100)

One-click: clone the repo, train the **real EMERALD architecture** (full `crafter_smoke` dims) for 20k env steps on the GPU, and save the checkpoint + eval logs to your Google Drive. Then download `logdir/emerald_smoke` and compare it against your local DreamerV3 20k checkpoint with `harness/eval_overlay.py`.

**Before running:** Runtime → Change runtime type → **A100 GPU**.

Expected wall-clock on an A100: roughly ~1–2h for 20k steps.

In [ ]:
# 1. Confirm the GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2. Get the code (emerald_torch is standalone — only needs the crafter package)
%cd /content
!git clone https://github.com/JuliusSamwer/Nexus_v1.git 2>/dev/null || (cd Nexus_v1 && git pull -q)
%cd /content/Nexus_v1
!pip install -q crafter
print('ready')

In [ ]:
# 3. Mount Drive so checkpoints survive disconnects
from google.colab import drive
drive.mount('/content/drive')
import os
LOGDIR = '/content/drive/MyDrive/nexus/emerald_smoke'
os.makedirs(LOGDIR, exist_ok=True)
print('saving checkpoints + eval logs to:', LOGDIR)

In [ ]:
# 4. Train the 20k checkpoint (full EMERALD architecture). Prints loss + eval as it goes.
STEPS = 20000
!python3 -m emerald_torch.train --configs crafter_smoke --device cuda --logdir {LOGDIR} --steps {STEPS}

In [ ]:
# 5. Zip the logdir and download it (to run the comparison locally against DreamerV3)
import shutil
shutil.make_archive('/content/emerald_smoke', 'zip', LOGDIR)
from google.colab import files
files.download('/content/emerald_smoke.zip')

## Compare against DreamerV3 (locally)

Unzip `emerald_smoke.zip` into your local `logdir/`, then from the repo root:

```bash
python3 -m harness.eval_overlay \
    --run dreamer=logdir/crafter_smoke \
    --run emerald=logdir/emerald_smoke \
    --ref EMERALD=results/EMERALD.json \
    --out results/eval_overlay
```

Produces `results/eval_overlay.png` (score / per-achievement / the cliff / score-vs-step) plus a per-achievement table and the head-to-head score delta.

> **Note:** 20k steps is *early* for both models (EMERALD's published score ≈57 is at ~1M+ steps). Expect mostly shallow achievements in both — read this as a directional / trajectory comparison and a pipeline check, not a verdict. Transformers also start slower than DreamerV3's GRU, so an early EMERALD deficit wouldn't be surprising.